# 🏠 التنبؤ بأسعار العقارات (House Price Prediction — Regression)
### مشروع B2 — مسار تعلم الآلة (Machine Learning Track)

---
## 🎯 المشكلة التجارية (Business Problem)
منصة عقارات عايزة **أداة تقدير تلقائي لسعر البيت** (زي Zillow's Zestimate) عشان تساعد البائعين
والمشترين يحطّوا سعر عادل، وتسرّع التقييم بدل المثمّن البشري.

**نوع المشكلة:** انحدار (Regression) — التنبؤ بقيمة رقمية مستمرة (السعر).

## 📦 ما الذي يثبته المشروع
معالجة القيم المفقودة · **تصحيح الالتواء (Skewness) باللوغاريتم** · Pipeline احترافي ·
**الانحدار المنتظم (Ridge/Lasso)** · Gradient Boosting · **تحليل البواقي (Residual Analysis)** · تفسير المعاملات.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/Ahmedelgendyyy/data-science-portfolio"
PROJECT = "ml/b2_house_prices"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| الانحدار الخطي وافتراضاته | ISLR (ch.3) | الأساس — لازم تفهم قبل أي حاجة |
| **التواء الهدف + Log transform** | Feature Eng. for ML (Zheng) | الأسعار ملتوية يمين → اللوغاريتم بيحسّن النموذج |
| القيم المفقودة (Imputation) | Géron (ch.2) | بيانات الواقع دايماً ناقصة |
| **الانتظام (Ridge / Lasso)** | ISLR (ch.6) | منع الـ overfitting + اختيار الميزات |
| Pipeline & ColumnTransformer | Géron (ch.2) | منع التسريب وتنظيم المعالجة |
| مقاييس الانحدار (RMSE, MAE, R²) | ISLR (ch.3) | تقييم الخطأ بوحدات السعر الحقيقية |
| **تحليل البواقي (Residuals)** | ISLR (ch.3) | كشف هل النموذج متحيّز أو ناقص ميزات |

> 🎯 **بيُستخدم في الواقع:** تقييم العقارات، تسعير السيارات المستعملة، تقدير قيمة الأصول، التأمين.


## 0️⃣ المكتبات

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [ ]:
df = pd.read_csv('data/house_prices_data.csv')
# TODO: اطبع shape والقيم المفقودة، واستعرض describe()
print(df.shape)

In [ ]:
corr = df.corr(numeric_only=True)['sale_price'].sort_values(ascending=False)
print('Correlation with price:\n', corr.round(2))
fig, ax = plt.subplots(1,2, figsize=(13,4))
sns.scatterplot(data=df, x='square_footage', y='sale_price', ax=ax[0], alpha=.4)
ax[0].set_title('Price vs Square Footage')
sns.boxplot(data=df, x='neighborhood', y='sale_price', ax=ax[1]); ax[1].set_title('Price by Neighborhood')
plt.tight_layout(); plt.show()

## 2️⃣ تصحيح التواء الهدف (Target Skewness → Log) 📐
أسعار العقارات **ملتوية لليمين** (قليل من البيوت غالية جداً). الانحدار الخطي بيفترض توزيع متماثل،
فبنطبّق `log1p` على السعر — ده بيحسّن الأداء بشكل ملحوظ. (نرجّع بـ `expm1` عند التقييم).

In [ ]:
# TODO: قارن توزيع السعر قبل وبعد log1p (اطبع الـ skew)
...

## 3️⃣ هندسة الميزات والمعالجة (Pipeline)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# TODO: اعمل ميزة house_age، خُد y = log1p(sale_price)
# TODO: ColumnTransformer = imputer+scaler للأرقام، imputer+onehot للفئات
df['house_age'] = ...
X = ...; y = ...
pre = ColumnTransformer([...])

## 4️⃣ مقارنة الموديلات بالـ Cross-Validation (RMSE بوحدات السعر)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_predict
# TODO: عرّف rmse_price ترجع الخطأ بوحدات السعر (expm1)
# TODO: قارن Linear/Ridge/Lasso/RF/XGBoost بـ 5-fold CV
def rmse_price(yt, yp): ...
models = {...}

## 5️⃣ تدريب أفضل موديل وتقييمه على الاختبار

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
# TODO: درّب أفضل موديل، تنبّأ، ارجع للسعر بـ expm1، واطبع RMSE/MAE/R²
best = ...

## 6️⃣ تحليل البواقي (Residual Analysis) 🔍
البواقي = (الحقيقي − المتوقع). المفروض تكون **عشوائية حول الصفر**. أي نمط فيها = الموديل بيفوّت حاجة.

In [ ]:
resid = true - pred
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].scatter(pred, resid, alpha=.4); ax[0].axhline(0, color='red', ls='--')
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Residual'); ax[0].set_title('Residuals vs Predicted')
sns.histplot(resid, kde=True, ax=ax[1]); ax[1].set_title('Residual distribution')
plt.tight_layout(); plt.show()

## 7️⃣ أهمية الميزات (Feature Importance)

In [ ]:
feat = best.named_steps['pre'].get_feature_names_out()
imp = pd.Series(best.named_steps['m'].feature_importances_, index=feat).sort_values()
imp.plot(kind='barh', figsize=(7,4), title='XGBoost Feature Importance'); plt.tight_layout(); plt.show()

## 8️⃣ الخلاصة والتوصيات (Conclusion)
- **النتيجة:** XGBoost حقّق أقل RMSE وأعلى R² — يقدر يقدّر السعر بدقة كويسة.
- **أهم العوامل:** المساحة (square footage)، الحي (neighborhood)، عدد الحمامات — متوافق مع منطق السوق.
- **اللوغاريتم:** تحويل الهدف حسّن الأداء وخلّى البواقي أقرب للتوزيع الطبيعي.
- **التوصية:** استخدام الموديل كأداة تقدير أولي + مراجعة بشرية للحالات الشاذة (بواقي كبيرة).
- **الخطوة القادمة:** إضافة ميزات الموقع الدقيق، المسافة للخدمات، وبيانات السوق الزمنية.

> ✅ **اللي اتعلمته:** EDA، تصحيح الالتواء، Imputation، Pipeline، الانتظام، GBM، وتحليل البواقي.
